In [26]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.ml.regression import DecisionTreeRegressor, RandomForestRegressor
from pyspark.ml.pipeline import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

import mlflow
from mlflow.models.signature import infer_signature
from mlflow.tracking import MlflowClient

import os
import pandas as pd

In [4]:
spark = SparkSession.builder.appName("treeairbnb").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/22 16:53:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
file_path = "/app/src/main/data/sf_airbnb/sf-airbnb-clean.parquet/"
os.path.exists(file_path)

data = spark.read.parquet(file_path)
data.select(
    "neighbourhood_cleansed",
    "room_type",
    "bedrooms",
    "bathrooms",
    "number_of_reviews",
    f.log("price").alias("price"),
)

DataFrame[neighbourhood_cleansed: string, room_type: string, bedrooms: double, bathrooms: double, number_of_reviews: double, price: double]

In [6]:
traindf, testdf = data.randomSplit([0.8, 0.2], seed=42)
print(
    f"There are {traindf.count()} rows in the training set and {testdf.count()} rows in the test set"
)


There are 5780 rows in the training set and 1366 rows in the test set


In [ ]:
categorical_cols = [field for (field, dtype) in traindf.dtypes if dtype == "string"]

index_output_cols = [x + "Index" for x in categorical_cols]
ohe_output_cols = [x + "OHE" for x in categorical_cols]

string_indexer = StringIndexer(
    inputCols=categorical_cols, outputCols=index_output_cols, handleInvalid="skip"
)
ohe_encoder = OneHotEncoder(inputCols=index_output_cols, outputCols=ohe_output_cols)

numerical_cols = [
    field
    for (field, dtype) in traindf.dtypes
    if (dtype == "double") & (field != "price")
]

assemble_inputs = index_output_cols + numerical_cols
vec_assembler = VectorAssembler(inputCols=assemble_inputs, outputCol="features")

dt = DecisionTreeRegressor(labelCol="price")
dt.setMaxBins(40)

rf = RandomForestRegressor(labelCol="price", maxBins=40, seed=42)

In [ ]:
stages = [string_indexer, vec_assembler, rf]
pipeline = Pipeline(stages=stages)
pipeline_model = pipeline.fit(traindf)
dtmodel = pipeline_model.stages[-1]

feature_imp = pd.DataFrame(
    list(zip(vec_assembler.getInputCols(), dtmodel.featureImportances)),
    columns=["feature", "importance"],
)
feature_imp.sort_values(by="importance", ascending=False)

,feature,importance
10,accommodates,0.141341
1,cancellation_policyIndex,0.128434
12,bedrooms,0.123870
3,neighbourhood_cleansedIndex,0.111406
8,latitude,0.097640
13,beds,0.081530
4,property_typeIndex,0.074584
11,bathrooms,0.060642
14,minimum_nights,0.045775
5,room_typeIndex,0.026790


In [28]:
param_grid = (
    ParamGridBuilder()
    .addGrid(rf.maxDepth, [2, 4, 6, 8])
    .addGrid(rf.numTrees, [10, 100, 200])
    .build()
)

In [29]:
evaluator = RegressionEvaluator(
    predictionCol="prediction", labelCol="price", metricName="rmse"
)

In [30]:
cv = CrossValidator(
    estimator=rf,
    evaluator=evaluator,
    estimatorParamMaps=param_grid,
    numFolds=5,
    seed=42,
    parallelism=4,
)

In [31]:
stages = [string_indexer, vec_assembler, cv]
pipeline = Pipeline(stages=stages)

In [32]:
with mlflow.start_run(run_name="random-forest") as run:
    pipeline_model = pipeline.fit(traindf)
    input_example = traindf.limit(1).toPandas()
    signature = infer_signature(traindf.toPandas())
    mlflow.spark.log_model(
        pipeline_model,
        "model",
        input_example=input_example,
        signature=signature,
        registered_model_name="AirbnbRandomForest",
    )

    cv_model = pipeline_model.stages[-1]
    best_rf_model = cv_model.bestModel

    mlflow.log_param("num_trees", best_rf_model.getNumTrees)
    mlflow.log_param("max_depth", best_rf_model.getMaxDepth)

    # log metrics
    predf = pipeline_model.transform(testdf)
    rmse = evaluator.evaluate(predf)
    r2 = evaluator.setMetricName("r2").evaluate(predf)
    mlflow.log_metrics({"rmse": rmse, "r2": r2})

    feat_imp = pd.DataFrame(
        list(zip(vec_assembler.getInputCols(), best_rf_model.featureImportances)),
        columns=["Features", "Importance"],
    ).sort_values(by="Importance", ascending=False)

    feat_imp.to_csv("feature-importance.csv")
    mlflow.log_artifact("feature-importance.csv")

2025/07/22 17:51:39 INFO mlflow.spark: File '/tmp/tmps88bulun/model/sparkml' is already on DFS, copy is not necessary.
Registered model 'AirbnbRandomForest' already exists. Creating a new version of this model...
Created version '2' of model 'AirbnbRandomForest'.


In [33]:
client = MlflowClient()
runs = client.search_runs(
    run.info.experiment_id, order_by=["attributes.start_time desc"], max_results=1
)

run_id = runs[0].info.run_id
runs[0].data.metrics

{'r2': 0.19574149132665952, 'rmse': 215.8754634163306}

In [36]:
load_model = mlflow.spark.load_model(f"mlruns/0/{run_id}/artifacts/model")

2025/07/22 18:05:03 INFO mlflow.spark: File 'mlruns/0/16abd716a5f64268a625890602e6da94/artifacts/model/sparkml' is already on DFS, copy is not necessary.
